In [9]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "russell2008image")
original_data_pathway = os.path.join(pathway, "original_data")

# complete_path_1 = os.path.join(original_data_pathway, "Russell_2008_BEHAV_PROCESS_IMES.sav")
complete_path_1 = os.path.join(original_data_pathway, "russell2008image_standardized.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [10]:
import pandas as pd
import numpy as np
import pyreadstat

# df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
df = pd.read_csv(complete_path_1)

# df['study_id']="russell2008image"
# df.columns = map(str.lower, df.columns)
# df=df.applymap(lambda s: s.lower() if type(s) == str else s)

In [11]:
# df.rename(columns={"species": "species_original", 
#     "name":"ape", 
#     "birthyr":"ape_birth_year"}, inplace=True)
# df['year']="2004"

In [12]:
# df['row_id'] = np.arange(len(df))
# df_a = df.copy()
# df_b = df.copy() 
# df_a.rename(columns={"nice20_1": "1", 
#         "nice20_2": "2", 
#         "nice20_3": "3", 
#         "nice20_4": "4",
#         "nice20_5":"5",
#         "nicet20":"average_percentage_of_time_spent_at_window_across_trials"}, inplace=True)

# df_a = df_a.melt(id_vars=["study_id", "row_id", 'year', "ape", 'ape_birth_year'],
#                           value_vars=["1", "2", "3", "4", "5", "average_percentage_of_time_spent_at_window_across_trials"],
#                           var_name="trial", value_name="nice20")

# df_b.rename(columns={"nast20_1": "1", 
#         "nast20_2": "2", 
#         "nast20_3": "3", 
#         "nast20_4": "4",
#         "nast20_5":"5",
#         "nastt20":"average_percentage_of_time_spent_at_window_across_trials"}, inplace=True)

# df_b = df_b.melt(id_vars=["study_id", "row_id", 'year', "ape", 'ape_birth_year'],
#                           value_vars=["1", "2", "3", "4", "5", "average_percentage_of_time_spent_at_window_across_trials"],
#                           var_name="trial", value_name="nasty20")

# fulldf = pd.concat([df_a, df_b], axis=1, join='inner')
# fulldf = fulldf.loc[:,~fulldf.columns.duplicated()]

In [13]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

# comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
# apedf = pd.read_csv(comp_path_ape_info)   
# fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

In [14]:
# fulldf.columns
# fulldf.rename(columns={"ape": "participant"}, inplace=True)

In [15]:

df.rename(columns={"year": "year_temp"}, inplace=True)
df[['year','month','day' ]] = df['test_date'].str.split('.',expand=True)

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') 
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'].replace('nan-nan-nan', np.nan, inplace=True, regex=True)
df['dodc'] = pd.to_datetime(df['dodc'])
df['dob'] = pd.to_datetime(df['dob'])

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365

df=df.sort_values(by = ['participant','trial'])

df.rename(columns={"nice20": "nice_percentage",
                   'nasty20':'nasty_percentage'}, inplace=True)

df = df[~df.trial.str.contains("5")]

In [16]:
russell2008image_standardized=df[['study_id', 'year','month','day', 'participant', 'age_in_years',  'sex', 'species', 'trial',
                                   'nice_percentage', 'nasty_percentage', 'first_shown']]
comp_out_path_stand = os.path.join(out_pathway, 'russell2008image_standardized.csv')
russell2008image_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =russell2008image_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
russell2008image_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'russell2008image_glossary.csv')
russell2008image_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

